In [1]:
import json
import os
import sys
import uuid

import requests
from dotenv import load_dotenv

load_dotenv(override=True)

# ====== CONFIG ======
BASE_URL = "http://localhost:7001"

TENANT_ID     = os.getenv("AZ_TENANT_ID",     "<tenant-id>")
CLIENT_ID     = os.getenv("AZ_CLIENT_ID",     "<client-id>")
OAUTH_SECRET = os.getenv("AZ_CLIENT_SECRET", "<client-secret>")
OAUTH_SCOPE   = os.getenv("AZ_OAUTH_SCOPE",   "3db474b9-6a0c-4840-96ac-1fceb342124f/.default")
ISSUER_API_URL = os.getenv("ISSUER_API_URL", "https://<host>/<path>")
DID_AUTHORITY = os.getenv("DID_AUTHORITY", "<did-authority>")
MANIFEST = os.getenv("MANIFEST", "<manifest>")
CALLBACK_URL = os.getenv("CALLBACK_URL", "<callback-url>")
# ===================


# check if container is up and running
try:
    response = requests.get(BASE_URL, timeout=5)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Connection error: {e}")

In [2]:
''' Create the issuance request URI '''

def _normalize_scope(scope: str) -> str:
    s = scope.strip()
    # add /.default if missing
    if not s.endswith("/.default"):
        s = s.rstrip("/") + "/.default"
    return s

def get_oauth_token(tenant_id, client_id, client_secret, scope):
    """
    OAuth2 client_credentials verso il token endpoint v2 of Microsoft
    """
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    data = {
        "client_id": client_id,
        "client_secret": client_secret,
        "grant_type": "client_credentials",
        "scope": _normalize_scope(scope),
    }
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    resp = requests.post(url, data=data, headers=headers, timeout=20)
    try:
        resp.raise_for_status()
    except requests.HTTPError:
        raise SystemExit(f"[TOKEN ERROR] {resp.status_code} {resp.text}")
    token = resp.json().get("access_token")

    if not token:
        raise SystemExit("[TOKEN ERROR] access_token missing in the response!")
    return token

def vc_issue(endpoint: str, callback_url: str, did: str, manifest: str, oauth_token: str, claims: dict):
    """ Make the issuance request to MS endpoint, then extract and return the request URI (to be passed to the agent) """
    headers = {
        "Authorization": f"Bearer {oauth_token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    payload = {
        "includeQRCode": False,
        "callback": {
            "url": callback_url,
            "state": str(uuid.uuid4()),
            "headers": {
                "api-key": "callback-api-key-SPIKEREPLY2025"
            }
        },
        "authority": did,
        "registration": {
            "clientName": "MS Entra Verified ID"
        },
        "type": "AdminCredential",
        "manifest": manifest,
        "claims": claims
    }

    try:
        resp = requests.post(
            url=endpoint,
            headers=headers,
            data=json.dumps(payload),
            timeout=30
        )
        # HTTP 201: success
        if resp.status_code != 201:
            resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        sys.exit(f"VC Issuance failed: {e}")

    # return something like openid-vc://?request_uri=https://verifiedid.did.msidentity.com/v1.0/tenants/{TENANT-ID}/verifiableCredentials/issuanceRequests/...
    return resp.json()


resp = vc_issue(
    endpoint=ISSUER_API_URL,
    callback_url=CALLBACK_URL,
    did=DID_AUTHORITY,
    manifest=MANIFEST,
    oauth_token=get_oauth_token(tenant_id=TENANT_ID, client_id=CLIENT_ID, client_secret=OAUTH_SECRET, scope=OAUTH_SCOPE),
    claims={"admin": "true"}    # "true" or "false" NOTHING ELSE
)

request_id = resp.get("requestId")
url = resp.get("url")

resp

{'requestId': '0db573c0-e3f3-4043-a0a2-247d095657e9',
 'url': 'openid-vc://?request_uri=https://verifiedid.did.msidentity.com/v1.0/tenants/070bd82a-c7ba-4bed-bc50-3dca476dc820/verifiableCredentials/issuanceRequests/0db573c0-e3f3-4043-a0a2-247d095657e9',
 'expiry': 1771250614}

In [3]:
# TODO: FILL WITH WALTID AGENT DATA
'''Create and login to walt.id account, then get the wallet ID'''

body = {
  "type": "email",
  "name": "Alice",
  "email": "alice@walt.id",
  "password": "aaa"
}

resp = requests.post(
    BASE_URL + "/wallet-api/auth/register",
    headers={"Content-Type": "application/json"},
    json=body,
    timeout=15
)


body = {
  "type": "email",
  "email": "alice@walt.id",
  "password": "aaa"
}

resp = requests.post(
    BASE_URL + "/wallet-api/auth/login",
    headers={"Content-Type": "application/json"},
    json=body,
    timeout=15
)

# if authN is successful, then save the token
waltid_token = resp.json().get("token") if resp.status_code == 200 else None


resp = requests.get(
    BASE_URL + "/wallet-api/wallet/accounts/wallets",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {waltid_token}"
    },
    timeout=15
)

# if code = 200 get the wallet id
wallet_id = resp.json().get("wallets")[0].get("id") if resp.status_code == 200 else None

wallet_id   # '9c9dea06-969a-4a13-86a1-9b10630a00a8'

'9c9dea06-969a-4a13-86a1-9b10630a00a8'

In [4]:
''' Exchange the credentials from microsoft createIssuanceRequest '''
# callbacks events are generated here

resp = requests.post(
    BASE_URL + f"/wallet-api/wallet/{wallet_id}/exchange/useOfferRequest",
    headers={
        "Content-Type": "text/plain",
        "Authorization": f"Bearer {waltid_token}"
    },
    data=url,
    timeout=15
)

resp.json()

[{'wallet': '9c9dea06-969a-4a13-86a1-9b10630a00a8',
  'id': '250c0f11-56e0-461f-8bf3-2fdf7ef99393',
  'document': 'eyJhbGciOiJFUzI1NiIsImtpZCI6ImRpZDp3ZWI6c3RnYWNjdGVzdGRpZC56Mzgud2ViLmNvcmUud2luZG93cy5uZXQjNjFkM2NhNjQ0N2E1NDk5ZTg4NDI5MjllMTRlZjUwODN2Y1NpZ25pbmdLZXktZGQ1ZjkiLCJ0eXAiOiJKV1QifQ.eyJ2YyI6eyJAY29udGV4dCI6WyJodHRwczovL3d3dy53My5vcmcvMjAxOC9jcmVkZW50aWFscy92MSJdLCJ0eXBlIjpbIlZlcmlmaWFibGVDcmVkZW50aWFsIiwiQWRtaW5DcmVkZW50aWFsIl0sImNyZWRlbnRpYWxTdWJqZWN0Ijp7ImFkbWluIjoidHJ1ZSJ9LCJjcmVkZW50aWFsU3RhdHVzIjp7ImlkIjoidXJuOnV1aWQ6NGE4YWM1MGQtNTJkNy00YTRjLWIzZWItOTIyM2IwYmE2MGI4P2JpdC1pbmRleD0zOSIsInR5cGUiOiJSZXZvY2F0aW9uTGlzdDIwMjFTdGF0dXMiLCJzdGF0dXNMaXN0SW5kZXgiOjM5LCJzdGF0dXNMaXN0Q3JlZGVudGlhbCI6ImRpZDp3ZWI6c3RnYWNjdGVzdGRpZC56Mzgud2ViLmNvcmUud2luZG93cy5uZXQ_c2VydmljZT1JZGVudGl0eUh1YiZxdWVyaWVzPVczc2liV1YwYUc5a0lqb2lRMjlzYkdWamRHbHZibk5SZFdWeWVTSXNJbk5qYUdWdFlTSTZJbWgwZEhCek9pOHZkek5wWkM1dmNtY3ZkbU10YzNSaGRIVnpMV3hwYzNRdE1qQXlNUzkyTVNJc0ltOWlhbVZqZEVsa0lqb2lOR0U0WVdNMU1HUXROVEprTn

In [5]:
''' Check the callback webhook to assess that the issuance was successful '''

resp = requests.get(
     CALLBACK_URL + f"/events/{request_id}",
    headers={
        "Content-Type": "application/json",
    },
    timeout=15
)

resp.json()

{'request_id': '0db573c0-e3f3-4043-a0a2-247d095657e9',
 'events': [{'id': 258,
   'request_id': '0db573c0-e3f3-4043-a0a2-247d095657e9',
   'status': 'request_retrieved',
   'timestamp': '2026-02-16 13:58:37',
   'state': '4989255b-c28b-4d5a-a095-b8ace9efc791',
   'subject': None,
   'verifiedCredentialsData': None},
  {'id': 259,
   'request_id': '0db573c0-e3f3-4043-a0a2-247d095657e9',
   'status': 'issuance_successful',
   'timestamp': '2026-02-16 13:58:38',
   'state': '4989255b-c28b-4d5a-a095-b8ace9efc791',
   'subject': None,
   'verifiedCredentialsData': None},
  {'id': 260,
   'request_id': '0db573c0-e3f3-4043-a0a2-247d095657e9',
   'status': 'request_retrieved',
   'timestamp': '2026-02-16 13:58:39',
   'state': '4989255b-c28b-4d5a-a095-b8ace9efc791',
   'subject': None,
   'verifiedCredentialsData': None}]}

In [6]:
''' List the credentials '''

resp = requests.get(
    BASE_URL + f"/wallet-api/wallet/{wallet_id}/credentials",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {waltid_token}"
    },
    timeout=15
)

resp.json()

i = 0
for item in resp.json():
    i+=1
    print(item)

print(i)

{'wallet': '9c9dea06-969a-4a13-86a1-9b10630a00a8', 'id': '250c0f11-56e0-461f-8bf3-2fdf7ef99393', 'document': 'eyJhbGciOiJFUzI1NiIsImtpZCI6ImRpZDp3ZWI6c3RnYWNjdGVzdGRpZC56Mzgud2ViLmNvcmUud2luZG93cy5uZXQjNjFkM2NhNjQ0N2E1NDk5ZTg4NDI5MjllMTRlZjUwODN2Y1NpZ25pbmdLZXktZGQ1ZjkiLCJ0eXAiOiJKV1QifQ.eyJ2YyI6eyJAY29udGV4dCI6WyJodHRwczovL3d3dy53My5vcmcvMjAxOC9jcmVkZW50aWFscy92MSJdLCJ0eXBlIjpbIlZlcmlmaWFibGVDcmVkZW50aWFsIiwiQWRtaW5DcmVkZW50aWFsIl0sImNyZWRlbnRpYWxTdWJqZWN0Ijp7ImFkbWluIjoidHJ1ZSJ9LCJjcmVkZW50aWFsU3RhdHVzIjp7ImlkIjoidXJuOnV1aWQ6NGE4YWM1MGQtNTJkNy00YTRjLWIzZWItOTIyM2IwYmE2MGI4P2JpdC1pbmRleD0zOSIsInR5cGUiOiJSZXZvY2F0aW9uTGlzdDIwMjFTdGF0dXMiLCJzdGF0dXNMaXN0SW5kZXgiOjM5LCJzdGF0dXNMaXN0Q3JlZGVudGlhbCI6ImRpZDp3ZWI6c3RnYWNjdGVzdGRpZC56Mzgud2ViLmNvcmUud2luZG93cy5uZXQ_c2VydmljZT1JZGVudGl0eUh1YiZxdWVyaWVzPVczc2liV1YwYUc5a0lqb2lRMjlzYkdWamRHbHZibk5SZFdWeWVTSXNJbk5qYUdWdFlTSTZJbWgwZEhCek9pOHZkek5wWkM1dmNtY3ZkbU10YzNSaGRIVnpMV3hwYzNRdE1qQXlNUzkyTVNJc0ltOWlhbVZqZEVsa0lqb2lOR0U0WVdNMU1HUXROVEprTnkwMFl

In [7]:
''' List the DIDs '''

resp = requests.get(
    BASE_URL + f"/wallet-api/wallet/{wallet_id}/dids",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {waltid_token}"
    },
    timeout=15
)

resp.json()

[{'did': 'did:jwk:eyJrdHkiOiJFQyIsImNydiI6IlAtMjU2Iiwia2lkIjoiZDQ5UFAzQ2oycnQwX0Rha2stN3VvQzFpS1I5QXNrQkpIN1lrTFFXRW9vUSIsIngiOiJFQzZIRzhOWlZLSy1EaVVQY3NkWktnUGxxbUpFMVVHZmJsbkRwSHBjcTZrIiwieSI6IjBfV1lrN2ljMk12ekd1WWJtOWJBcjM0Vkxkb2F4VndscVZUTjVIMmhlRmMifQ',
  'alias': 'Onboarding',
  'document': '{"@context":["https://www.w3.org/ns/did/v1","https://w3id.org/security/suites/jws-2020/v1"],"id":"did:jwk:eyJrdHkiOiJFQyIsImNydiI6IlAtMjU2Iiwia2lkIjoiZDQ5UFAzQ2oycnQwX0Rha2stN3VvQzFpS1I5QXNrQkpIN1lrTFFXRW9vUSIsIngiOiJFQzZIRzhOWlZLSy1EaVVQY3NkWktnUGxxbUpFMVVHZmJsbkRwSHBjcTZrIiwieSI6IjBfV1lrN2ljMk12ekd1WWJtOWJBcjM0Vkxkb2F4VndscVZUTjVIMmhlRmMifQ","verificationMethod":[{"id":"did:jwk:eyJrdHkiOiJFQyIsImNydiI6IlAtMjU2Iiwia2lkIjoiZDQ5UFAzQ2oycnQwX0Rha2stN3VvQzFpS1I5QXNrQkpIN1lrTFFXRW9vUSIsIngiOiJFQzZIRzhOWlZLSy1EaVVQY3NkWktnUGxxbUpFMVVHZmJsbkRwSHBjcTZrIiwieSI6IjBfV1lrN2ljMk12ekd1WWJtOWJBcjM0Vkxkb2F4VndscVZUTjVIMmhlRmMifQ#0","type":"JsonWebKey2020","controller":"did:jwk:eyJrdHkiOiJFQyIsImNydiI6IlAtMj